In [1]:
5+5

10

In [1]:
import sys
from pathlib import Path


import torch 
import torch.nn as nn
import torch.nn.functional as F


# Notebook is at notebooks/, so go up 1 level to the repository root
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(REPO_ROOT)

D:\GAN_AI\From Scratch\Bilingual_GPT_from_scratch


In [2]:
torch.manual_seed(0)

In [3]:
from tokenize import Token

from src.llm.model.attention import MultiHeadAttention
from src.llm.model.feedforward import FeedForward
from src.llm.config import Modelconfig

TOKENS = ["the" , "cat" , "sat" , "on" , "mat"]

config = Modelconfig(
    vocab_size= 32,
    d_model=8,
    n_heads=2,
    max_seq_len=16,
    n_layer=2,
    dropout=0.1
)

B = 1 
T = len(TOKENS)

x = torch.randn(B , T , config.d_model)

In [4]:
x

tensor([[[-1.1258, -1.1524, -0.2506, -0.4339,  0.8487,  0.6920, -0.3160,
          -2.1152],
         [ 0.3223, -1.2633,  0.3500,  0.3081,  0.1198,  1.2377,  1.1168,
          -0.2473],
         [-1.3527, -1.6959,  0.5667,  0.7935,  0.5988, -1.5551, -0.3414,
           1.8530],
         [-0.2159, -0.7425,  0.5627,  0.2596, -0.1740, -0.6787,  0.9383,
           0.4889],
         [ 1.2032,  0.0845, -1.2001, -0.0048, -0.5181, -0.3067, -1.5810,
           1.7066]]])

In [5]:
attention = MultiHeadAttention(config)
ffn = FeedForward(config)

In [16]:
att_out , updated_cache = attention(x)
y = ffn(att_out)

In [17]:
x.shape , att_out.shape , y.shape

(torch.Size([1, 5, 8]), torch.Size([1, 5, 8]), torch.Size([1, 5, 8]))

In [18]:
ln = nn.LayerNorm(config.d_model)

In [19]:
dig_layer = torch.tensor(
    [
        [
            [50.0 , 52.0 , 100.0 , 40.0 ,59.0 , 72.0 , 400.0 , 65.0]
        ] 
    ]
)

In [20]:
ln(dig_layer)

tensor([[[-0.4851, -0.4674, -0.0421, -0.5737, -0.4053, -0.2902,  2.6159,
          -0.3522]]], grad_fn=<NativeLayerNormBackward0>)

# GPT model

In [21]:
from src.llm.model.GPT_block import GPTBlock

In [22]:
attention = MultiHeadAttention(config)
ffn = FeedForward(config)
att_out , updated_cache = attention(x)
y = ffn(att_out)

In [23]:
blocks = nn.ModuleList(
    GPTBlock(config) for i in range(config.n_layer)
)

In [24]:
blocks

ModuleList(
  (0-1): 2 x GPTBlock(
    (mha): MultiHeadAttention(
      (W_q): Linear(in_features=8, out_features=8, bias=False)
      (W_k): Linear(in_features=8, out_features=8, bias=False)
      (W_v): Linear(in_features=8, out_features=8, bias=False)
      (W_o): Linear(in_features=8, out_features=8, bias=True)
    )
    (ff): FeedForward(
      (fc1): Linear(in_features=8, out_features=3072, bias=True)
      (act): GELU(approximate='tanh')
      (fc2): Linear(in_features=3072, out_features=8, bias=True)
    )
    (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True, bias=True)
    (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [25]:
ln_final = nn.LayerNorm(config.d_model)
lm_head = nn.Linear(config.d_model , config.vocab_size , bias=False)

# init weights

In [26]:
import math

def _init_weights(module):
    """GPT-2's initialisation: normal(0, 0.02), zeroed biases.

        PyTorch's Linear default depends on fan-in, which leaves a 768-wide
        model's activations larger than GPT-2 assumes and makes the early
        steps of training less stable.
    """

    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weights, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weights, mean=0.0, std=0.02)

def _scale_residual_projections(blocks, config):

    scale = math.sqrt( 2 *  config.n_layers)
    for block in blocks:
        with torch.no_grad():
            block.attn.W_o.weight.div_(scale)
            block.ffn.fc2.weight.div_(scale)

# The loss funcation

In [27]:
sentence = torch.tensor([[5, 9, 14, 3, 21, 7]])

inputs = sentence[:, 0:-1]
targets = sentence[:, 1:]

print("sentence:", sentence.tolist()[0])
print("inputs  :", inputs.tolist()[0])
print("targets :", targets.tolist()[0])

print()
print(f"{'position':>9}{'reads':>8}{'must predict':>14}")
for t in range(inputs.size(1)):
    print(f"{t:>9}{inputs[0, t].item():>8}{targets[0, t].item():>14}")

sentence: [5, 9, 14, 3, 21, 7]
inputs  : [5, 9, 14, 3, 21]
targets : [9, 14, 3, 21, 7]

 position   reads  must predict
        0       5             9
        1       9            14
        2      14             3
        3       3            21
        4      21             7


GPT test

In [6]:
from src.llm.model.GPT_model import GPTModel

In [7]:
config.GPT_Small()

Modelconfig(vocab_size=32000, d_model=512, d_ff=2048, max_seq_len=1024, dropout=0.1, n_heads=8, n_layer=8, qkv_bias=False, pad_id=0, tie_weights=True)

In [8]:
gpt_model = GPTModel(config.GPT_Small())

In [9]:
gpt_model.num_parameters()

42116096